# Event Selection

**_must run with dfs will all reconstructed slices and tracks_**

This notebook runs the full event selection, evaluating selection performance

- makes mode breakdown bar plots for each selection stage
- makes purity/efficiency summary plots

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import sys
from os import path, makedirs
from datetime import datetime
import pickle

# local imports
# sys.path.append('../../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.categories import *
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.files_config_new import *
from analysis_village.numucc_1p0pi.makedf.selections import *
from pyanalib.split_df_helpers_new import *
from pyanalib.pandas_helpers import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use("presentation.mplstyle")

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

OSError: 'presentation.mplstyle' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)

In [ ]:
show_plot = True


save_fig = True
save_fig_base_dir = "/exp/sbnd/data/users/munjung/xsec/PLOTS/numuCC_1p0pi"
today_str = datetime.now().strftime("%Y%m%d")
today_str = "thesis"
save_fig_dir = path.join(save_fig_base_dir, f"event_selection-{today_str}")
if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)


save_content = True
save_content_base_dir = "/exp/sbnd/data/users/munjung/xsec/RESULTS/numuCC_1p0pi"
save_content_dir = path.join(save_content_base_dir, f"event_selection-{today_str}")
if save_content:
    if not path.exists(save_content_dir):
        makedirs(save_content_dir)
    print("saving nevts in ", save_content_dir)

saving plots in  /exp/sbnd/data/users/munjung/xsec/PLOTS/numuCC_1p0pi/event_selection-20260503
saving nevts in  /exp/sbnd/data/users/munjung/xsec/RESULTS/numuCC_1p0pi/event_selection-20260503


In [ ]:
# MC
df_dir_mc = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_all-mc-BNB_cosmics"
filename_str_mc = "sel_all-mc-BNB_cosmics_100"
keys2load_mc = ['evt', 'trk', 'hdr'] #'hit0', 'hit1', 'hit2',
df_mc = dfs_from_dir(search_dir=df_dir_mc, filename_str=filename_str_mc, keys2load=keys2load_mc, n_max_concat=999)

# Data
df_dir_data = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102054__sel_all-data-BNB_cosmics"
filename_str_data = "sel_all-data-BNB_cosmics_1"
keys2load_data = ['evt', 'trk', 'hdr', 'bnbpot', 'trigger']
df_data = dfs_from_dir(search_dir=df_dir_data, filename_str=filename_str_data, keys2load=keys2load_data, n_max_concat=999)

# LowE Dirt
df_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/MC/lowE"
keys2load_lowE = ['hdr', 'evt', 'trk']
df_lowE = dfs_from_dir(search_dir=df_dir, filename_str="aa_all", keys2load=keys2load_lowE, n_max_concat=999)

# OffBeam
df_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/data/OffBeam"
keys2load_offbeam = ['hdr', 'evt', 'trk']
df_offbeam = dfs_from_dir(search_dir=df_dir, filename_str="_all", keys2load=keys2load_offbeam, n_max_concat=999)

# Intime
df_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/MC/intime"
keys2load_intime = ['hdr', 'evt', 'trk']
df_intime = dfs_from_dir(search_dir=df_dir, filename_str="aa_all", keys2load=keys2load_intime, n_max_concat=999)

Found 11 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_all-mc-BNB_cosmics/sel_all-mc-BNB_cosmics_100.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_all-mc-BNB_cosmics/sel_all-mc-BNB_cosmics_1000.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_all-mc-BNB_cosmics/sel_all-mc-BNB_cosmics_1001.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_all-mc-BNB_cosmics/sel_all-mc-BNB_cosmics_1002.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_all-mc-BNB_cosmics/sel_all-mc-BNB_cosmics_1003.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_all-mc-BNB_cosmics/sel_all-mc-BNB_cosmics_1004.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_all-mc-BNB_cosmics/sel_all-mc-BNB_cosmics_1005.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102644__sel_

100%|██████████| 11/11 [00:14<00:00,  1.36s/it]


Found 111 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102054__sel_all-data-BNB_cosmics/sel_all-data-BNB_cosmics_1.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102054__sel_all-data-BNB_cosmics/sel_all-data-BNB_cosmics_10.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102054__sel_all-data-BNB_cosmics/sel_all-data-BNB_cosmics_100.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102054__sel_all-data-BNB_cosmics/sel_all-data-BNB_cosmics_101.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102054__sel_all-data-BNB_cosmics/sel_all-data-BNB_cosmics_102.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102054__sel_all-data-BNB_cosmics/sel_all-data-BNB_cosmics_103.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_01_102054__sel_all-data-BNB_cosmics/sel_all-data-BNB_cosmics_104.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/202

 50%|█████     | 56/111 [02:04<01:40,  1.82s/it]/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/pyanalib/split_df_helpers_new.py:105: RuntimeWarning: overflow encountered in scalar add
  ntuple_offset += ntuple_vals.max() + 1
 51%|█████▏    | 57/111 [02:05<01:34,  1.75s/it]/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/pyanalib/split_df_helpers_new.py:105: RuntimeWarning: overflow encountered in scalar add
  ntuple_offset += ntuple_vals.max() + 1
 54%|█████▍    | 60/111 [02:18<02:45,  3.25s/it]/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/pyanalib/split_df_helpers_new.py:105: RuntimeWarning: overflow encountered in scalar add
  ntuple_offset += ntuple_vals.max() + 1
 55%|█████▍    | 61/111 [02:20<02:24,  2.89s/it]/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/pyanalib/split_df_helpers_new.py:105: RuntimeWarning: overflow encountered in scalar add
  ntuple_offset += ntuple_vals.max() + 1
 57%|█████▋    | 63/111 [02:24<01:54,  2.39s/it]/exp/sbnd/app/users/munjung/xsec/freeze/cafp

Found 1 files to process
Files to process: ['/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/MC/lowE/aa_all.df']


100%|██████████| 1/1 [00:26<00:00, 26.22s/it]


Found 4 files to process
Files to process: ['/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/data/OffBeam/aa_all.df', '/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/data/OffBeam/ab_all.df', '/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/data/OffBeam/ac_all.df', '/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09/data/OffBeam/test_all.df']


 25%|██▌       | 1/4 [03:30<10:32, 210.81s/it]


KeyboardInterrupt: 

In [ ]:
# -- Assign event and track DataFrames for each sample
mc_hdr_df, mc_evt_df, mc_trk_df = df_mc["hdr"], df_mc["evt"], df_mc["trk"]
data_hdr_df, data_evt_df, data_trk_df = df_data["hdr"], df_data["evt"], df_data["trk"]
dirt_hdr_df, dirt_evt_df, dirt_trk_df = df_lowE["hdr"], df_lowE["evt"], df_lowE["trk"]
offbeam_hdr_df, offbeam_evt_df, offbeam_trk_df = df_offbeam["hdr"], df_offbeam["evt"], df_offbeam["trk"]
intime_hdr_df, intime_evt_df, intime_trk_df = df_intime["hdr"], df_intime["evt"], df_intime["trk"]

# -- Exposure calculation for BNB data
data_tot_pot = data_hdr_df['pot'].sum()
pot_str = get_pot_str(data_tot_pot)
data_evt_df["pot_weight"] = 1.0  # np.ones(len(data_evt_df))
data_trk_df["pot_weight"] = 1.0  # np.ones(len(data_trk_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print(f"data_tot_pot: {data_tot_pot:.3e}", f"data tot gates : {data_gates:.3e}")

# -- BNB MC scaling
mc_tot_pot = mc_hdr_df['pot'].sum()
mc_pot_scale = data_tot_pot / mc_tot_pot if mc_tot_pot > 0 else 0
mc_evt_df["pot_weight"] = mc_pot_scale
mc_trk_df["pot_weight"] = mc_pot_scale
print(f"mc_tot_pot: {mc_tot_pot:.3e}", f"mc_pot_scale: {mc_pot_scale:.3e}")

# -- Dirt sample scaling
dirt_tot_pot = dirt_hdr_df['pot'].sum()
dirt_pot_scale = data_tot_pot / dirt_tot_pot if dirt_tot_pot > 0 else 0
dirt_evt_df["pot_weight"] = dirt_pot_scale
dirt_trk_df["pot_weight"] = dirt_pot_scale
print(f"dirt_tot_pot: {dirt_tot_pot:.3e}", f"dirt_pot_scale: {dirt_pot_scale:.3e}")

# -- Offbeam exposure scaling
f = 0.08
offbeam_gates = offbeam_hdr_df.loc[offbeam_hdr_df['first_in_subrun'] == 1, 'noffbeambnb'].sum()
scale_offbeam_to_lightdata = (1-f) * data_gates / offbeam_gates if offbeam_gates > 0 else 0
offbeam_evt_df["gates_weight"] = scale_offbeam_to_lightdata
offbeam_evt_df["pot_weight"] = scale_offbeam_to_lightdata
offbeam_trk_df["pot_weight"] = scale_offbeam_to_lightdata
print(f"offbeam cosmics data gates: {offbeam_gates:.2e}", f"goal scale: {scale_offbeam_to_lightdata:.2f}")

# -- Intime MC cosmics scaling
intime_gates = intime_hdr_df.loc[intime_hdr_df['first_in_subrun'] == 1, 'ngenevt'].sum()
scale_intime_to_lightdata = (1-f) * data_gates / intime_gates if intime_gates > 0 else 0
intime_evt_df["gates_weight"] = scale_intime_to_lightdata
intime_evt_df["pot_weight"] = scale_intime_to_lightdata
intime_trk_df["pot_weight"] = scale_intime_to_lightdata
print(f"intime cosmics data gates: {intime_gates:.2e}", f"goal scale: {scale_intime_to_lightdata:.2f}")

In [ ]:
pot_str = get_pot_str(data_tot_pot)
plot_labels_bar = ["Events (POT={})".format(pot_str), "", ""]
plot_labels_hist = ["", "Events (POT={})".format(pot_str), ""]

In [ ]:
def plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=True):

    ret_dict = {}

    for bar_type in ["topology", "genie"]: # "nu_cosmics"
        save_name = save_fig_dir + "/bar_plot-{}-{}.png".format(bar_type, stage_key)
        this_ret = bar_plot(breakdown_type=bar_type,
                            mc_df=mc_df, intime_df=intime_df, dirt_df=dirt_df,
                            show_plot=show_plot, plot_labels=plot_labels_bar,
                            save_fig=save_fig, save_name=save_name)
        ret_dict[bar_type] = this_ret

    return ret_dict

# Event Selection

In [ ]:
# save breakdowns for summary plots
breakdown_dict = {"topology": {}, "genie": {}}

# save dfs per stage, df_dict used for efficiency plot
df_dict = {} 
df_dict_data = {}
df_dict_intime = {} 
df_dict_offbeam = {}
df_dict_dirt = {}

## Cosmic Rejection

In [3]:
stage_key = "allreco"

mc_df     = mc_evt_df
data_df   = data_evt_df
intime_df = intime_evt_df
offbeam_df = offbeam_evt_df
dirt_df   = dirt_evt_df

for d, d_dict in zip(
    (mc_df, data_df, intime_df, offbeam_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_offbeam, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

NameError: name 'mc_evt_df' is not defined

In [ ]:
stage_key = "is_clear_cosmic"

mc_df, data_df, intime_df, dirt_df = (
    cut_clear_cosmic(df)
    for df in (mc_df, data_df, intime_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

In [ ]:
stage_key = "vertex_in_fv"

mc_df, data_df, intime_df, offbeam_df, dirt_df = (
    cut_vertex_in_fv(df, det=DETECTOR)
    for df in (mc_df, data_df, intime_df, offbeam_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, offbeam_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_offbeam, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

In [ ]:
var_config = VariableConfig.nu_score()
plot_labels = ["Neutrino Score", "Events (POT={})".format(pot_str), ""]

for plot_type in ["topology"]:
    save_name = save_fig_dir + "/selected-{}_{}.png".format("nu_score", plot_type)
    ret_hist_topo = overlay_hists(plot_type,
                                  mc_df=mc_df, 
                                  data_df=data_df, 
                                  intime_df=intime_df, 
                                  dirt_df=dirt_df,
                                  ratio=True,
                                  ax_ylim_ratio=1.8,
                                  vline=[[0.45, 1]],
                                  var_config=var_config,
                                  plot_labels=plot_labels,
                                  save_fig=save_fig, 
                                  save_name=save_name)

In [ ]:
stage_key = "nu_score"

mc_df, data_df, intime_df, offbeam_df, dirt_df = (
    cut_nu_score(df, NU_SCORE_TH)
    for df in (mc_df, data_df, intime_df, offbeam_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, offbeam_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_offbeam, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

## Slice has two tracks

In [ ]:
# TODO: this is for det var samples
# TODO: move to dfmaker
# def get_chi2_avg(trkdf, pid_branch):
#     avg_val = avg_chi2(mc_trk_df, pid_branch)
#     trkdf[("pfp", "trk", "chi2pid", "avg", pid_branch, "")] = avg_val
#     return trkdf

# mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = (
#     get_chi2_avg(df, "chi2_muon")
#     for df in (mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df)
# )

# mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = (
#     get_chi2_avg(df, "chi2_proton")
#     for df in (mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df)
# )

In [ ]:
chimu_avg = avg_chi2(mc_trk_df, "chi2_muon")
mc_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_muon", "")] = chimu_avg
chip_avg = avg_chi2(mc_trk_df, "chi2_proton")
mc_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_proton", "")] = chip_avg

chimu_avg = avg_chi2(data_trk_df, "chi2_muon")
data_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_muon", "")] = chimu_avg
chip_avg = avg_chi2(data_trk_df, "chi2_proton")
data_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_proton", "")] = chip_avg

chimu_avg = avg_chi2(intime_trk_df, "chi2_muon")
intime_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_muon", "")] = chimu_avg
chip_avg = avg_chi2(intime_trk_df, "chi2_proton")
intime_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_proton", "")] = chip_avg

chimu_avg = avg_chi2(dirt_trk_df, "chi2_muon")
dirt_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_muon", "")] = chimu_avg
chip_avg = avg_chi2(dirt_trk_df, "chi2_proton")
dirt_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_proton", "")] = chip_avg

In [ ]:
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = (
    get_valid_trks(df)
    for df in (mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df)
)

mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = (
    match_trkdf_to_slcdf(trk_df, df)
    for trk_df, df in ((mc_trk_df, mc_df), (data_trk_df, data_df), (intime_trk_df, intime_df), (offbeam_trk_df, offbeam_df), (dirt_trk_df, dirt_df))
)

mc_df, data_df, intime_df, offbeam_df, dirt_df = (
    get_trk_info(df, trk_df, SAVE_NTRKS)
    for df, trk_df in ((mc_df, mc_trk_df), (data_df, data_trk_df), (intime_df, intime_trk_df), (offbeam_df, offbeam_trk_df), (dirt_df, dirt_trk_df))
)

In [ ]:
var_config = VariableConfig.n_trks()
plot_labels = ["Number of tracks", "Events (POT={})".format(pot_str), ""]

for plot_type in ["topology"]:
    save_name = save_fig_dir + "/selected-{}_{}.png".format("ntrks", plot_type)
    ret_hist_topo = overlay_hists(plot_type,
                                  mc_df=mc_df, 
                                  data_df=data_df, 
                                  intime_df=intime_df, 
                                  dirt_df=dirt_df,
                                  var_config=var_config,
                                  plot_labels=plot_labels,
                                  save_fig=save_fig, 
                                  save_name=save_name)

    save_name = save_fig_dir + "/selected-{}_{}.png".format("ntrks", plot_type)
    ret_hist_topo = overlay_hists(plot_type,
                                  mc_df=mc_df, 
                                  data_df=data_df, 
                                  intime_df=offbeam_df, 
                                  dirt_df=dirt_df,
                                  var_config=var_config,
                                  plot_labels=plot_labels,
                                  save_fig=save_fig, 
                                  save_name=save_name)

In [ ]:
stage_key = "2prong"

mc_df, data_df, intime_df, offbeam_df, dirt_df = (
    cut_2prong(df)
    for df in (mc_df, data_df, intime_df, offbeam_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, offbeam_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_offbeam, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

In [ ]:
stage_key = "2prong-contained"

mc_df, data_df, intime_df, offbeam_df, dirt_df = (
    cut_2prong_contained(df, det=DETECTOR)
    for df in (mc_df, data_df, intime_df, offbeam_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, offbeam_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_offbeam, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

In [ ]:
plot_type = "pdg"

mc_trk_df = pd.concat([mc_df.trk1, mc_df.trk2])
data_trk_df = pd.concat([data_df.trk1, data_df.trk2])
intime_trk_df = pd.concat([intime_df.trk1, intime_df.trk2])
offbeam_trk_df = pd.concat([offbeam_df.trk1, offbeam_df.trk2])
dirt_trk_df = pd.concat([dirt_df.trk1, dirt_df.trk2])

var_config = VariableConfig.track_score()

plot_labels = [var_config.var_labels[0], "Tracks / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(plot_type,
                         mc_df=mc_trk_df, 
                         data_df=data_trk_df, 
                         intime_df=intime_trk_df, 
                         dirt_df=dirt_trk_df,
                         syst=None,
                         syst_decomp = False,
                         ratio=True,
                         ax_ylim_ratio=1.8,
                         vline=[[0.5, 1]],
                         var_config=var_config,
                         plot_labels=plot_labels,
                         save_fig=save_fig, 
                         save_name=save_name)

In [ ]:
stage_key = "2prong-trackscore"

mc_df, data_df, intime_df, offbeam_df, dirt_df = (
    cut_2prong_trackscore(df, TRACKSCORE_TH)
    for df in (mc_df, data_df, intime_df, offbeam_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

In [ ]:
plot_type = "pdg"

mc_trk_df = pd.concat([mc_df.trk1, mc_df.trk2])
data_trk_df = pd.concat([data_df.trk1, data_df.trk2])
intime_trk_df = pd.concat([intime_df.trk1, intime_df.trk2])
offbeam_trk_df = pd.concat([offbeam_df.trk1, offbeam_df.trk2])
dirt_trk_df = pd.concat([dirt_df.trk1, dirt_df.trk2])

var_config = VariableConfig.vtx_dist()

plot_labels = [var_config.var_labels[0], "Tracks / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(plot_type,
                                mc_df=mc_trk_df, 
                                data_df=data_trk_df, 
                                intime_df=intime_trk_df, 
                                dirt_df=dirt_trk_df,
                                syst=None,
                                syst_decomp=False,
                                ratio=True,
                                vline=[[VTXDIST_TH, 0]],
                                var_config=var_config,
                                plot_labels=plot_labels,
                                save_fig=save_fig, 
                                save_name=save_name)

In [ ]:
stage_key = "2prong-vtxdist"

mc_df, data_df, intime_df, offbeam_df, dirt_df = (
    cut_2prong_vtxdist(df, VTXDIST_TH)
    for df in (mc_df, data_df, intime_df, offbeam_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, offbeam_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_offbeam, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, offbeam_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

## PID

In [ ]:
mc_trk_df_ = mc_trk_df.copy()
data_trk_df_ = data_trk_df.copy()
intime_trk_df_ = intime_trk_df.copy()
offbeam_trk_df_ = offbeam_trk_df.copy()
dirt_trk_df_ = dirt_trk_df.copy()

In [ ]:
# all tracks in 2-track slices
plot_type = "pdg"

mc_trk_df = pd.concat([mc_df.trk1, mc_df.trk2])
data_trk_df = pd.concat([data_df.trk1, data_df.trk2])
intime_trk_df = pd.concat([intime_df.trk1, intime_df.trk2])
dirt_trk_df = pd.concat([dirt_df.trk1, dirt_df.trk2])
            
var_config = VariableConfig.trk_len()
plot_labels = [var_config.var_labels[0], "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(plot_type,
                                mc_df=mc_trk_df, 
                                data_df=data_trk_df, 
                                intime_df=offbeam_trk_df, 
                                dirt_df=dirt_trk_df,
                                ratio=True,
                                # vline=[MU_CHI2MU_TH],
                                var_config=var_config,
                                plot_labels=plot_labels,
                                save_fig=False, 
                                save_name=save_name)

# with open(f'{save_content_dir}/{syst_tag}_{var_config.var_save_name}.pkl', 'wb') as f:
#     pickle.dump(ret_hist, f)


In [ ]:
def get_mcs_range_diff(trks):
    mcs_range_diff = (trks.pfp.trk.rangeP.p_muon - trks.pfp.trk.mcsP.fwdP_muon) / trks.pfp.trk.rangeP.p_muon
    trks[("pfp", "trk", "mcs_range_diff", "", "", "")] = mcs_range_diff
    return trks


mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = (
    get_mcs_range_diff(df)
    for df in (mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df)
)

In [ ]:
for var_config in [VariableConfig.mcs_range_diff()]: 

    plot_labels = [var_config.var_labels[0], "Events (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
    ret_hist_chi2mu = overlay_hists(plot_type,
                                    mc_df=mc_trk_df, 
                                    data_df=data_trk_df, 
                                    intime_df=offbeam_trk_df, 
                                    dirt_df=dirt_trk_df,
                                    ratio=True,
                                    vline=[[-QUAL_TH,0], [QUAL_TH,1]],
                                    var_config=var_config,
                                    plot_labels=plot_labels,
                                    save_fig=False, 
                                    save_name=save_name)

    # with open(f'{save_content_dir}/{syst_tag}_{var_config.var_save_name}-len50cm.pkl', 'wb') as f:
    #     pickle.dump(ret_hist, f)

In [ ]:
var_config = VariableConfig.chi2_mu()
plot_labels = [var_config.var_labels[0], "Tracks/ Bin  (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist_chi2mu = overlay_hists(plot_type,
                                mc_df=mc_trk_df, 
                                data_df=data_trk_df, 
                                intime_df=offbeam_trk_df, 
                                dirt_df=dirt_trk_df,
                                syst=None,
                                syst_decomp = False,
                                ratio=True,
                                vline=[[MU_CHI2MU_TH, 0]],
                                var_config=var_config,
                                plot_labels=plot_labels,
                                save_fig=save_fig, 
                                save_name=save_name)

var_config = VariableConfig.chi2_proton()
plot_labels = [var_config.var_labels[0], "Tracks / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist_chi2mu = overlay_hists(plot_type,
                                mc_df=mc_trk_df, 
                                data_df=data_trk_df, 
                                intime_df=offbeam_trk_df, 
                                dirt_df=dirt_trk_df,
                                syst=None,
                                syst_decomp = False,
                                ratio=True,
                                vline=[[MU_CHI2P_TH, 1]],
                                ax_ylim_ratio=1.8,
                                var_config=var_config,
                                plot_labels=plot_labels,
                                save_fig=save_fig, 
                                save_name=save_name)


In [ ]:
# compare chi2_mu vs. chi2_p 2D distributions and get the percentage out of total of the selected area
from matplotlib.colors import LogNorm

var_config_1 = VariableConfig.chi2_mu()
var_config_2 = VariableConfig.chi2_proton()

plt.hist2d(mc_trk_df[var_config_1.var_evt_reco_col], 
           mc_trk_df[var_config_2.var_evt_reco_col], 
           bins=[var_config_1.bins, var_config_2.bins],
           cmap="GnBu", norm=LogNorm())
plt.colorbar(label="Tracks")
plt.xlabel("Muon $\\chi^2$")
plt.ylabel("Proton $\\chi^2$")
# plt.title("Shorter Track of Selected 2-Track Events")
plt.text(0.025, 1.07, r"$\mathbf{SBND}$ Preliminary    $\mathbf{SBND}$ Simulation", transform=plt.gca().transAxes, fontsize=14, color='gray', ha='left', va='top')
if save_fig:
    save_name = save_fig_dir + "/chi2muon_vs_chi2proton-MC.pdf"
    plt.savefig(save_name, bbox_inches="tight")
plt.show()

plt.hist2d(data_trk_df[var_config_1.var_evt_reco_col], 
           data_trk_df[var_config_2.var_evt_reco_col], 
           bins=[var_config_1.bins, var_config_2.bins],
           cmap="GnBu", norm=LogNorm())
plt.colorbar(label="Tracks")
plt.xlabel("Muon $\\chi^2$")
plt.ylabel("Proton $\\chi^2$")
# plt.title("Shorter Track of Selected 2-Track Events")
plt.text(0.025, 1.07, r"$\mathbf{SBND}$ Preliminary    $\mathbf{SBND}$ Data", transform=plt.gca().transAxes, fontsize=14, color='gray', ha='left', va='top')
if save_fig:
    save_name = save_fig_dir + "/chi2muon_vs_chi2proton-data.pdf"
    plt.savefig(save_name, bbox_inches="tight")
plt.show()

In [ ]:
# muon quality cuts

# tracks after len > 50 cm cut for muon selection
def len_cut(trks):
    trks = trks[trks.pfp.trk.len > 50]
    return trks

# mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = (
#     len_cut(df)
#     for df in (mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df)
# )


# tracks after quality cut for muon selection
def mcs_range_diff_cut(trks):
    trks = trks[np.abs(trks.pfp.trk.mcs_range_diff) < QUAL_TH]
    return trks

# mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = (
#     mcs_range_diff_cut(df)
#     for df in (mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df)
# )

def is_mu_candidate(trks):
    trks = len_cut(trks)
    trks = mcs_range_diff_cut(trks)
    chimu_avg = trks.pfp.trk.chi2pid.avg.chi2_muon
    chip_avg = trks.pfp.trk.chi2pid.avg.chi2_proton
    chi2_cut = (chimu_avg > 0) & (chimu_avg < MU_CHI2MU_TH) & (chip_avg > MU_CHI2P_TH) 
    trks = trks[chi2_cut]
    return trks

def is_not_mu_candidate(trks):
    nlevels = len(mc_df.index.names)
    mcs_range_diff = np.abs((trks.pfp.trk.rangeP.p_muon - trks.pfp.trk.mcsP.fwdP_muon) / trks.pfp.trk.rangeP.p_muon)
    chimu_avg = trks.pfp.trk.chi2pid.avg.chi2_muon
    chip_avg = trks.pfp.trk.chi2pid.avg.chi2_proton
    mu_cut = (chimu_avg > 0) & (chimu_avg < MU_CHI2MU_TH) & \
            (chip_avg > MU_CHI2P_TH) & \
            (trks.pfp.trk.len > MU_LEN_TH) & \
            (mcs_range_diff < QUAL_TH)
    not_mu_candidate = pd.concat([trks[~mu_cut], trks[mu_cut].groupby(level=list(range(nlevels))).nth(1)])
    return not_mu_candidate


In [ ]:
# get the percentage of tracks selected as mu

mc_mus = is_mu_candidate(mc_trk_df)
intime_mus = is_mu_candidate(intime_trk_df)
dirt_mus = is_mu_candidate(dirt_trk_df)
data_mus = is_mu_candidate(data_trk_df)

n_mc_mus = len(mc_mus.groupby(level=[0,1,2]).head(1))
n_intime_mus = len(intime_mus.groupby(level=[0,1,2]).head(1))
n_dirt_mus = len(dirt_mus.groupby(level=[0,1,2]).head(1))
n_data_mus = len(data_mus.groupby(level=[0,1,2]).head(1))

print((n_mc_mus + n_intime_mus + n_dirt_mus) / (len(mc_trk_df) + len(intime_trk_df) + len(dirt_trk_df) ))
print(n_data_mus / len(data_trk_df))


In [ ]:
# tracks that aren't muon candidates
plot_type = "pdg"

mc_notmus = is_not_mu_candidate(mc_trk_df)
data_notmus = is_not_mu_candidate(data_trk_df)
intime_notmus = is_not_mu_candidate(intime_trk_df)
dirt_notmus = is_not_mu_candidate(dirt_trk_df)

for var_config in [VariableConfig.chi2_mu(), 
                   VariableConfig.chi2_proton()]:

    plot_labels = [var_config.var_labels[0], "Events (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
    ret_hist = overlay_hists(plot_type,
                                    mc_df=mc_notmus, 
                                    data_df=data_notmus, 
                                    intime_df=intime_notmus, 
                                    dirt_df=dirt_notmus,
                                    ratio=True,
                                    # vline=[MU_CHI2MU_TH],
                                    var_config=var_config,
                                    plot_labels=plot_labels,
                                    save_fig=False, 
                                    save_name=save_name)

    # with open(f'{save_content_dir}/{syst_tag}_{var_config.var_save_name}-not_mu_candidate.pkl', 'wb') as f:
    #     pickle.dump(ret_hist, f)

In [ ]:
mc_df, data_df, intime_df, dirt_df = (
    get_mu_p_candidate(df, 
                       mu_chi2mu_th=MU_CHI2MU_TH, mu_chi2p_th=MU_CHI2P_TH, mu_len_th=MU_LEN_TH, qual_th=QUAL_TH,
                       p_chi2mu_th=-1, p_chi2p_th=P_CHI2P_TH, p_len_th=P_LEN_TH)
    for df in (mc_df, data_df, intime_df, dirt_df)
)

In [ ]:
stage_key = "2prong-muX"

mc_df, data_df, intime_df, dirt_df = (
    cut_has_mu(df)
    for df in (mc_df, data_df, intime_df, dirt_df)
)
mc_df, data_df, intime_df, dirt_df = (
    cut_mu_kinematics(df, mu_Plo_th=MU_PLO_TH, mu_Phi_th=MU_PHI_TH)
    for df in (mc_df, data_df, intime_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

In [ ]:
stage_key = "2prong-mup"

mc_df, data_df, intime_df, dirt_df = (
    cut_has_p(df)
    for df in (mc_df, data_df, intime_df, dirt_df)
)
mc_df, data_df, intime_df, dirt_df = (
    cut_p_kinematics(df, p_Plo_th=P_PLO_TH, p_Phi_th=P_PHI_TH)
    for df in (mc_df, data_df, intime_df, dirt_df)
)

for d, d_dict in zip(
    (mc_df, data_df, intime_df, dirt_df),
    (df_dict, df_dict_data, df_dict_intime, df_dict_dirt)
    ):
    d_dict[stage_key] = d

ret = plot_bar_plots(stage_key, mc_df, intime_df, dirt_df, show_plot=show_plot)
for key in ["topology", "genie"]:
    breakdown_dict[key][stage_key] = ret[key]["perc_list"]

In [ ]:
sel_dfs = {
    "mc_df": mc_df,
    "data_df": data_df,
    "intime_df": intime_df,
    "dirt_df": dirt_df
}

# with open(f'{save_content_dir}/{syst_tag}_sel_mup.pkl', 'wb') as f:
#     pickle.dump(sel_dfs, f)


In [ ]:
save_content_dir

# Event Selection Summary Breakdown Plot

In [ ]:
print(df_dict.keys())
stage_labels = [
    "All reconstructed slices",
    "Not clear cosmic",
    "Vertex in fiducial volume",
    "Nu-score > {}".format(NU_SCORE_TH),
    "Has exactly 2 PFPs",
    "Both PFPs contained",
    "Both PFPs have track score > {}".format(TRACKSCORE_TH),
    "Both track \n(start position - vertex) < {} cm".format(VTXDIST_TH),
    "One track is muon-like",
    "The other is proton-like"
]

In [ ]:
stages = list(breakdown_dict["topology"].keys())[::-1]
y = np.arange(len(stages))
bar_width = 0.3

def stack_bars(ax, data, yoffset, colors, label):
    left = np.zeros(len(stages))
    bars = []
    for i, color in enumerate(colors[:data.shape[1]]):
        b = ax.barh(y + yoffset, data[:, i], bar_width, left=left, color=color, label=label if i == 0 else None)
        bars.append(b)
        left += data[:, i]
    return bars

topo_data = np.array([breakdown_dict["topology"][stage] for stage in stages])[:, ::-1]
genie_data = np.array([breakdown_dict["genie"][stage] for stage in stages])[:, ::-1]

fig, ax = plt.subplots(figsize=(10, 10))
stack_bars(ax, topo_data, -bar_width/2, topology_colors, "Topology")
stack_bars(ax, genie_data,  bar_width/2,  genie_mode_colors, "GENIE")

ax.set_xlabel("Percentage (%)")
ax.set_yticks(y)
ax.set_yticklabels(stage_labels[::-1], fontsize=12)

common_patches = [Patch(facecolor=c, label=l) for c, l in zip(
    ["gray", "sienna", "crimson", "darkgreen"],
    ["Cosmic", r"Out FV $\nu$", r"In FV other $\nu$", r"In FV $\nu_{\mu}$ NC"]
)]
genie_patches = [Patch(facecolor=c, label=l) for c, l in zip(
    ["#BFB17C", "#D88A3B", "#2c7c94", "#390C1E", "#9b5580"],
    [r"In FV $\nu_{\mu}$ CC Other", r"In FV $\nu_{\mu}$ CC SIS/DIS", r"In FV $\nu_{\mu}$ CC RES", r"In FV $\nu_{\mu}$ CC MEC", r"In FV $\nu_{\mu}$ CC QE"]
)]
topo_patches = [Patch(facecolor=c, label=l) for c, l in zip(
    ["coral", "darkslateblue", "mediumslateblue"],
    [r"In FV $\nu_{\mu}$ CC Other", r"In FV $\nu_{\mu}$ CC Np0$\pi$", r"In FV $\nu_{\mu}$ CC 1p0$\pi$"]
)]

ax.legend(handles=common_patches, loc='upper left', bbox_to_anchor=(0.01,1.18), ncol=4, fontsize=12, frameon=False)
for i, handles in enumerate([genie_patches[::-1], topo_patches[::-1]]):
    ax_i = ax.twinx()
    ax_i.legend(handles=handles, loc='upper left', 
                bbox_to_anchor=(0.01, 1.14 - 0.07*i), # space out
                ncol=3 if i==0 else 4, fontsize=12, frameon=False)
    ax_i.set_yticks([])

if save_fig:
    plt.savefig(f"{save_fig_dir}/event_selection_summary.png", dpi=300, bbox_inches="tight")

plt.tight_layout()
plt.show()

In [ ]:
eps = 1e-8
ratio = True
approval = "internal"
textloc = [0.05, 0.55]
ax_ylim_ratio = 1.6
breakdown_type = "topology"

ret = overlay_hists(breakdown_type=breakdown_type,
                    var_config=VariableConfig.muon_momentum(),
                    mc_df=mc_df,
                    data_df=data_df,
                    intime_df=intime_df,
                    ax_ylim_ratio=ax_ylim_ratio,
                    ratio=ratio,
                    textloc=textloc,
                    approval=approval,
                    plot_labels=plot_labels_hist,
                    syst=None,
                    save_fig=False, 
                    save_name=None)

# Efficiency Curves

In [ ]:
eff_dict = {}
for var_config in [VariableConfig.muon_momentum(), VariableConfig.muon_direction(),
                   VariableConfig.proton_momentum(), VariableConfig.proton_direction()]:
                #    VariableConfig.opening_angle(),
                #    VariableConfig.tki_del_Tp(), VariableConfig.tki_del_p(),
                #    VariableConfig.tki_del_alpha(), VariableConfig.tki_del_phi(),]:

   save_name = save_fig_dir + "/efficiency-{}.png".format(var_config.var_save_name)
   ret = plot_efficiency(df_dict,
                   stage_labels,
                   var_config, 
                   textloc=[0.05, 1.08],
                   approval="internal", 
                   legend=False,
                   save_fig=save_fig, 
                   save_name=save_name)

   eff_dict[var_config.var_save_name] = ret

In [ ]:
var_config = VariableConfig.neutrino_energy()
save_name = save_fig_dir + "/efficiency-{}.png".format(var_config.var_save_name)
ret = plot_efficiency(df_dict,
                stage_labels,
                var_config, 
                textloc=[0.05, 1.08],
                approval="internal", 
                legend=True,
                save_fig=save_fig, 
                save_name=save_name)

eff_dict[var_config.var_save_name] = ret

In [ ]:
with open(f'{save_fig_dir}/{syst_tag}_eff_dict.pkl', 'wb') as f:
    pickle.dump(eff_dict, f)

In [ ]:
save_content_dir